[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarioSigal/TP_Rompecabezas/blob/main/TP_Rompecabezas_Colab.ipynb)

# 🧩 Trabajo Práctico: Resolución Automatizada de Rompecabezas
**Procesamiento de Imágenes (PDI)** Nivel 6

**Grupo:**

**Integrantes:**

---

## 🎯 Objetivo General
En este Ejercicio Final combinaran todo lo aprendido por separado en el colab anteriores. En este nivel cada Rompecabezas sera una combinacion de los niveles 1,2,3 y 4.

Cada rompecabezas podra contener como no:
- Ruido Global (Nivel 1)
- Alteracion Cromatica (Nivel 2)
- Ruido Periodico (Nivel 3)

Cada rompecabezas tendra las muecas del nivel 4.

### Suposiciones
- Nunca les tocara ruido Salt & Pepper cuando la pieza tiene ruido periodico.
- Cada pieza tendra el mismo ***tipo*** de alteracion cromatica, si el rompecabezas lo tiene.
- El orden de la aplicacion de ruidos es el siguiente:
  - Ruido Global
  - Alteracion Cromatica
  - Ruido Periodico


---
## ⚙️ Configuración del Entorno de Ejecución

Esta celda configura automáticamente las rutas necesarias tanto si se ejecuta en **Google Colab** como en un entorno local de Jupyter.


In [4]:
# Configuración de entorno para Google Colab y ejecución local
import os, sys
from pathlib import Path

if 'google.colab' in str(get_ipython()):
    print('--> Entorno detectado: Google Colab')
    if not os.path.exists('core'):
        !git clone https://github.com/MarioSigal/TP_Rompecabezas.git repo_tp
        %cd repo_tp
        !git checkout main
    sys.path.insert(0, os.getcwd())
else:
    print('--> Entorno detectado: Local')
    raiz = Path.cwd()
    if (raiz / 'core').exists():
        sys.path.insert(0, str(raiz))
    elif (raiz / 'TP_Rompecabezas' / 'core').exists():
        sys.path.insert(0, str(raiz / 'TP_Rompecabezas'))
    elif (raiz / 'TP_FINAL_ALUMNOS' / 'core').exists():
        sys.path.insert(0, str(raiz / 'TP_FINAL_ALUMNOS'))

import numpy as np
import cv2
import matplotlib.pyplot as plt

# Importar funciones del núcleo del TP
from core import (
    cargar_imagen,
    guardar_imagen,
    preparar_imagen_base,
    crear_rompecabezas_nivel,
    crear_dataset_desafio_30,
    reconstruir_rompecabezas,
    reconstruir_desde_afinidades,
    construir_matrices_afinidad,
    compatibilidad_baseline,
    generar_reporte_completo,
    imprimir_reporte,
)

from utils import (
    mostrar_piezas_desordenadas,
    mostrar_comparacion_imagen,
    mostrar_espectro_fourier,
    mostrar_reconstruccion,
    crear_animacion,
)

print('✅ ¡Módulos del TP cargados con éxito!')


--> Entorno detectado: Local
✅ ¡Módulos del TP cargados con éxito!


## Configuracion de imagenes

In [ ]:

#TODO: COMPLETAR ESTO CON LAS IMAGENES FINALES, CHEQUEAR QUE LAS USADAS SON BUENAS

VARIANTE_CROMATICA_POR_IMAGE={
    "IMAGEN_1.png" : "value",
    "IMAGEN_2.png" : "matiz",
    ...
}

SEMILLA_POR_IMAGEN={
    "IMAGEN_1.png" : 111,
    "IMAGEN_2.png" : 222,
    ...
}

### Creacion de Puzzles

In [ ]:
from pathlib import Path

CARPETA_DATASET = Path("content/imagenes/dataset_desafio")

rutas_imagenes = sorted(
        p for p in CARPETA_DATASET.iterdir()
        if p.suffix.lower() in EXTENSIONES_VALIDAS
    )


puzzles = []
for ruta_imagen in rutas_imagenes:
    imagen = cargar_imagen(ruta_imagen)
    variante_cromatica = VARIANTE_CROMATICA_POR_IMAGE[file_path.name]
    puzzle = crear_rompecabezas_nivel(
        imagen,
        nivel=6,
        filas=8,
        columnas=8,
        semilla=101,
        variante_cromatica= variante_cromatica
    )
    puzzles.append(puzzle)

## Definicion del mecanismo de Resolucion

In [5]:
def detectar_tipo_ruido(piezas: list) -> dict:
    """

    Retorna un diccionario con el diagnóstico, por ejemplo:
    {
        'tiene_sal': bool,
        'tiene_pimienta': bool,
        'tiene_continuo': bool,
        'tipo': str
    }
    """
    ###COMPLETAR

    return {'tiene_sal': True, 'tiene_pimienta': True, 'tiene_continuo': True, 'tipo': 'mixto'}


def filtrar_pieza(pieza: np.ndarray, diagnostico: dict = None) -> np.ndarray:
    """
    Limpia la pieza adaptando el filtro espacial según el diagnóstico detectado:
    - Si tiene sal y/o pimienta: ...
    - Si tiene ruido continuo: ...
    Recibe y retorna la pieza en float64 [0.0, 1.0] RGB.
    """
    ###COMPLETAR

    return pieza

In [6]:
def compatibilidad_baseline(
    pieza_a: np.ndarray,
    pieza_b: np.ndarray,
    relacion: str,
) -> float:
    """
    Calcula el costo de acople entre dos piezas usando el error cuadratico medio (Baseline).
    El cuadrado (en vez de valor absoluto) penaliza mucho mas fuerte los saltos grandes y
    casi no penaliza los chicos, lo que agudiza la separacion entre vecinos verdaderos
    (saltos chicos) y falsos (saltos grandes).
    Cuanto MENOR sea el resultado, mayor es la similitud de los bordes.

    Parámetros:
    pieza_a: np.ndarray
        Pieza base (origen).
    pieza_b: np.ndarray
        Pieza vecina propuesta.
    relacion: str
        'horizontal' (B a la derecha de A) o 'vertical' (B abajo de A).

    Retorna:
    float
        Valor de error/costo promedio entre las líneas externas de los bordes.

    Ejemplo de uso
    --------------
    >>> pieza_1 = np.ones((30, 30, 3)) * 100
    >>> pieza_2 = np.ones((30, 30, 3)) * 105
    >>> costo = compatibilidad_baseline(pieza_1, pieza_2, relacion="horizontal")
    >>> print(f"Costo de poner pieza_2 a la derecha de pieza_1: {costo:.1f}")
    Costo de poner pieza_2 a la derecha de pieza_1: 250.0
    """
    if relacion not in BORDES_ENFRENTADOS:
        raise ValueError(f"Relación inválida: '{relacion}'. Se espera 'horizontal' o 'vertical'.")

    lado_a, lado_b = BORDES_ENFRENTADOS[relacion]
    banda_a = extraer_banda_borde(pieza_a, lado_a, cantidad_lineas=1)
    banda_b = extraer_banda_borde(pieza_b, lado_b, cantidad_lineas=1)

    if banda_a.shape[0] != banda_b.shape[0]:
        n_samples = min(banda_a.shape[0], banda_b.shape[0])
        idx_a = np.linspace(0, banda_a.shape[0] - 1, n_samples).astype(int)
        idx_b = np.linspace(0, banda_b.shape[0] - 1, n_samples).astype(int)
        banda_a = banda_a[idx_a]
        banda_b = banda_b[idx_b]

    #Error cuadratico en vez de absoluto: penaliza mucho mas fuerte los saltos
    #grandes y casi no penaliza los chicos, lo que agudiza la separacion entre
    #vecinos verdaderos (saltos chicos) y falsos (saltos grandes)
    return float(np.linalg.norm(banda_a - banda_b)**2)

## Corrida Final

In [7]:
def get_reporte_por_puzzle(puzzle, 
                           funcion_compatibilidad=compatibilidad_baseline,
                           filtro_por_pieza = filtrar_pieza,
                           detector_tipo_ruido= detectar_tipo_ruido):
    
    piezas = puzzle.piezas
    diagnostico_de_ruido = detector_tipo_ruido(piezas)
    if diagnostico_de_ruido:
        piezas_limpias = [filtrar_pieza(p, diagnostico_de_ruido) for p in piezas]
        piezas = piezas_limpias

    matrices_afinidad = construir_matrices_afinidad(piezas, funcion_compatibilidad)
    grilla, rec = reconstruir_desde_afinidades(matrices_afinidad, puzzle.cantidad_filas, puzzle.cantidad_columnas, devolver_reconstructor=True)

    reporte = generar_reporte_completo(puzzle, matrices_afinidad=matrices_afinidad, grilla_propuesta=grilla)
    return reporte

In [ ]:
reportes = []
for puzzle in puzzles:
    reportes.append(get_reporte_por_puzzle(puzzle,
                                           funcion_compatibilidad=compatibilidad_baseline,
                                           filtro_por_pieza=filtrar_pieza,
                                           detector_tipo_ruido=detectar_tipo_ruido))

In [ ]:
def calcular_reporte_promedio(reportes: list) -> dict:
    """
    Promedia las metricas numericas (ver generar_reporte_completo) de una lista
    de reportes, uno por puzzle. Si algun reporte no tiene una clave (p.ej.
    'psnr'/'ssim' cuando no se paso imagen_limpiada), esa clave se promedia
    solo sobre los reportes que si la tienen.
    """
    if not reportes:
        return {}

    suma = {}
    conteo = {}
    for reporte in reportes:
        for clave, valor in reporte.items():
            if not isinstance(valor, (int, float)):
                continue
            suma[clave] = suma.get(clave, 0.0) + valor
            conteo[clave] = conteo.get(clave, 0) + 1

    return {clave: suma[clave] / conteo[clave] for clave in suma}


## Puntaje Total del la Reconstruccion

In [ ]:
calcular_reporte_promedio(reportes)